# Deep learning models for arrhythmia detection and diagnosis

## Initialisation

In [2]:
# Imports
from os.path import join as osj
import pandas as pd
import numpy as np
import torch
import wfdb
import pywt
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.signal import resample


# Directory where the dataset is stored
dataset_directory = 'mit-bih-arrhythmia-database-1.0.0/'

# Get an array of all ids
record_ids = pd.read_csv('mit-bih-arrhythmia-database-1.0.0/RECORDS', header=None).to_numpy().reshape(-1)

# Initialise dictionaries to store data
lead1 = {}
lead2 = {}
annotations = {}
sample_size = 300

# Information about the labels
beat_annotations = [
    'N',  # Normal beat
    'L',  # Left bundle branch block beat
    'R',  # Right bundle branch block beat
    'B',  # Bundle branch block beat (unspecified)
    'A',  # Atrial premature beat
    'a',  # Aberrated atrial premature beat
    'J',  # Nodal (junctional) premature beat
    'S',  # Supraventricular premature or ectopic beat (atrial or nodal)
    'V',  # Premature ventricular contraction
    'r',  # R-on-T premature ventricular contraction
    'F',  # Fusion of ventricular and normal beat
    'e',  # Atrial escape beat
    'j',  # Nodal (junctional) escape beat
    'n',  # Supraventricular escape beat (atrial or nodal)
    'E',  # Ventricular escape beat
    '/',  # Paced beat
    'f',  # Fusion of paced and normal beat
    'Q',  # Unclassifiable beat
    '?',  # Beat not classified 
]

non_beat_annotations = [
    '[',  # Start of ventricular flutter/fibrillation
    '!',  # Ventricular flutter wave
    ']',  # End of ventricular flutter/fibrillation
    'x',  # Non-conducted P-wave (blocked APC)
    '(',  # Waveform onset
    ')',  # Waveform end
    'p',  # Peak of P-wave
    't',  # Peak of T-wave
    'u',  # Peak of U-wave
    '`',  # PQ junction
    "'",  # J-point
    '^',  # Non-captured pacemaker artifact
    '|',  # Isolated QRS-like artifact
    '~',  # Change in signal quality
    '+',  # Rhythm change
    's',  # ST segment change
    'T',  # T-wave change
    '*',  # Systole
    'D',  # Diastole
    '=',  # Measurement annotation
    '"',  # Comment annotation
    '@'   # Link to external data
]

# Load ECG signals and annotations
for id in record_ids:
    signals, info = wfdb.io.rdsamp(osj(dataset_directory, str(id)))
    annotation = wfdb.rdann(dataset_directory + str(id), "atr")
    lead1[id] = signals[:, 0]
    lead2[id] = signals[:, 1]
    annotations[id] = annotation
    
# Create a Dataset object to be used later for training and testing
class ECGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(np.array(X), dtype=torch.float32) 
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


## Data Preprocessing

### Preprocessing functions for the beat Classifier

In [31]:
def swt_denoise(signal):
    """
    Denoise signal using Stationary Wavelet Transform (SWT) with soft thresholding.

    Parameters:
        signal (np.ndarray): The input 1D signal.
        wavelet (str): The wavelet to use.

    Returns:
        np.ndarray: The denoised signal.
    """
    # Get max level
    max_level = pywt.swt_max_level(len(signal))
    
    wavelet = "bior3.1"
    
    # Decompose
    coeffs = pywt.swt(signal, wavelet, level=max_level)
    
    # Estimate noise from first detail coefficients
    detail_coeffs = np.array(coeffs[0][1]) # First detail coefficients
    sigma = np.median(np.abs(detail_coeffs)) / 0.6745 # Median absolute deviation
    threshold = sigma * np.sqrt(2 * np.log(len(signal))) # Universal threshold
    
    # Apply soft thresholding to detail coefficients
    new_coeffs = []
    for approx, detail in coeffs:
        detail = pywt.threshold(detail, threshold, mode='soft')
        new_coeffs.append((approx, detail))
    
    # Reconstruct
    denoised = pywt.iswt(new_coeffs, wavelet)
    return denoised

def standardise(data):
    """
    Standardises the input.
    
    Parameters:
        data : np.ndarray
        
    Returns:
        standardised data : np.ndarray
    """
    mean_val = np.mean(data)
    std_val = np.std(data)
    return (data - mean_val) / std_val

def resize_segment(segment, target_length=sample_size):
    return resample(segment, target_length)

def segmentation_by_midpoint(signal, annotation, skew=0.66):  
    """ 
    OUTDATED BUT WORTH INCLUDING FOR REFERENCE, DO NOT USE.
    Segments the data from signal based on the locations in annotation and the skew
    
    Parameters:
        signal : np.ndarray
        annotation : an annotation object extracted from the .atr file
        skew : default = 0.66. a float that represents the percentage of samples taken after the annotation. e.g. 0.4 would mean 40% samples taken after and the rest are before.
        keep_first : default = False. If True, the first segment is kept as a segment.
        
    Returns:
        segments : list of np.ndarray
        labels : list of strings
    """
    r_peaks = annotation.sample
    r_labels = annotation.symbol
    
    segments = []
    labels = []
    
    # Ensure enough R-peaks to segment
    if len(r_peaks) < 2:
        return [], []
    
    # Compute midpoints between R-peaks
    midpoints = [
        int(r_peaks[i] + skew * (r_peaks[i + 1] - r_peaks[i]))
        for i in range(len(r_peaks) - 1)
    ]
    
    # Handle all beats with a midpoint either side (not first and last)
    for i in range(1, len(midpoints)-1):
        start = midpoints[i]
        end = midpoints[i+1]
        segment = signal[start:end]
        label = r_labels[i]  # Label corresponding to the center R-peak
        if label in beat_annotations:
            segments.append(segment)
            if label == 'N':
                labels.append(0) 
            else:
                labels.append(1)      
        else:
            continue
    
    # Handle the first beat
    if midpoints[0] > 0:
        first_label = r_labels[0]
        if first_label in beat_annotations:
            first_segment = signal[:midpoints[0]]
            segments.insert(0, first_segment)
            if first_label == 'N':
                labels.insert(0, 0)
            else:
                labels.insert(0, 1)

    # Handle the last beat
    if midpoints[-1] < len(signal):
        last_label = r_labels[-1]
        if last_label in beat_annotations:
            last_segment = signal[midpoints[-1]:]
            segments.append(last_segment)
            if last_label == 'N':
                labels.append(0)
            else:
                labels.append(1)


    segments = [resize_segment(seg) for seg in segments]
    
    return segments, labels
            
def segmentation_fixed_window(signal, annotation, before=100, after=200):
    """
    Segments beats using a fixed window around each R-peak.

    Parameters:
        signal (np.ndarray): 1D ECG signal.
        annotation (wfdb.Annotation): Annotation object with R-peaks.
        before (int): Number of samples before the R-peak.
        after (int): Number of samples after the R-peak.

    Returns:
        segments (List[np.ndarray]): List of 300-sample segments.
        labels (List[str]): Corresponding beat labels.
    """
    r_peaks = annotation.sample
    r_labels = annotation.symbol

    segments = []
    labels = []
    
    
    for peak, label in zip(r_peaks, r_labels):
        start = peak - before
        end = peak + after 

        # Ensure segment is within signal bounds
        if start < 0 or end > len(signal):
            continue
        
        if label in beat_annotations:
            if label == 'N':
                labels.append(0) 
            else:
                labels.append(1)
                
        else:
            continue

        segment = signal[start:end]
        segments.append(segment)

    return segments, labels
             
def process_signal(id):
    """
    Takes an id and returns the X, y for that id.
    
    Parameters:
        id (int): The id of the record.
        
    Returns:
        tuple: A tuple containing the segments and labels.
    """
    unfiltered, annotation = lead1[id], annotations[id]
    denoised_data = swt_denoise(unfiltered)
    normalised_data = standardise(denoised_data)
    X, y = segmentation_by_midpoint(normalised_data, annotation)
    
    X = np.stack(X) 
    X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  

    y = torch.tensor(y, dtype=torch.long)
    
    return X, y

### Preprocessing for Signal Classifier

### Apply processing to data and separate into train and testing

In [121]:
X = []
y = []

for id in record_ids:
    X_i, y_i = process_signal(id)
    X.append(X_i)
    y.append(y_i)
    
X = np.concatenate(X, axis=0)  # Shape: (total_samples, signal_length)
y = np.concatenate(y, axis=0)  # Shape: (total_samples,)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2) # stratify=y maybe

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25) 

In [ ]:
import random

plt.figure(figsize=(10, 4))
plt.plot(np.squeeze(X_train[random.randint(1,87557)]), color='blue')
plt.title("ECG Segment Example")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.grid(True)
plt.tight_layout()
plt.show()

## Models

### Beat Classifier

In [ ]:
class BeatClassifier(nn.Module):
    def __init__(self, input_size=sample_size, num_classes=2):
        super(BeatClassifier, self).__init__()
        
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=7, stride=1, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # output shape: (batch_size, 128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),  # flatten (batch_size, 128, 1) -> (batch_size, 128)
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 2)  # binary classification: normal vs abnormal
        )
    
    def forward(self, x):
        x = self.feature_extractor(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

### Signal Classifier

In [83]:
class SignalClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(SignalClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]  # Take the last time step output
        x = self.dropout(x)
        x = self.fc(x)
        return x
    
class SignalTransformer(nn.Module):
    def __init__(self, input_size=3, d_model=64, num_heads=4, num_layers=2, num_classes=5):
        super(SignalTransformer, self).__init__()
        
        # Positional Encoding
        self.positional_encoding = nn.Parameter(torch.zeros(1, 100, d_model))  # Max sequence length = 100
        
        # Transformer Encoder Layer
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=num_heads)
        self.transformer = nn.TransformerEncoder(self.encoder_layer, num_layers=num_layers)
        
        # Fully Connected Layer
        self.fc = nn.Linear(d_model, num_classes)
        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, x):
        # x: Batch size x sequence length x input size
        x = x * np.sqrt(64)  # Scaling input (similar to Transformer)
        
        # Add positional encoding (optional)
        x = x + self.positional_encoding[:, :x.size(1), :]
        
        # Transformer expects input shape (seq_len, batch_size, input_size)
        x = x.permute(1, 0, 2)  # Shape: (sequence_length, batch_size, feature_size)
        
        # Pass through Transformer Encoder
        x = self.transformer(x)
        
        # Use the output of the last token
        x = x[-1, :, :]  # Only take the last token's output
        
        # Final classification
        x = self.fc(x)
        x = self.softmax(x)
        return x

## Training

### Training Parameters

In [122]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

epochs = 10
batch_size = 64
learning_rate = 0.001

train_dataset = ECGDataset(X_train, y_train)
test_dataset = ECGDataset(X_test, y_test)
validation_dataset = ECGDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

beat_classifier = BeatClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimiser = optim.Adam(beat_classifier.parameters(), lr=learning_rate)

cpu


### Training loops

In [127]:
def train_beat_classifier(model, criterion, optimiser, train_dataloader, val_dataloader, epochs=10, device="cpu"):
    for epoch in range(epochs):
        # Training phase
        model.train()
        total_loss, correct, total = 0, 0, 0

        for inputs, labels in train_dataloader:
            inputs = inputs.to(device) 
            labels = labels.to(device)

            optimiser.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimiser.step()

            total_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_acc = correct / total
        
        # Validation phase
        model.eval()  # Set the model to evaluation mode
        val_loss, val_correct, val_total = 0, 0, 0
        
        with torch.no_grad():  # No gradients are computed during validation
            for inputs, labels in val_dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss / val_total
        val_acc = val_correct / val_total

        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.4f} - "
              f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")


def train_signal_classifier(model, criterion, optimizer):
    pass

## Evaluation

In [128]:
def test_model(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    print(f"Test Accuracy:     {acc*100:.4f}") # average=None? 
    print(f"Test Precision:    {precision*100:.4f}")
    print(f"Test Recall:       {recall*100:.4f}")
    print(f"Test F1 Score:     {f1*100:.4f}")

In [129]:
train_beat_classifier(beat_classifier, loss_function, optimiser, train_loader, validation_loader)

test_model(beat_classifier, test_loader)

Epoch 1/10 - Train Loss: 0.4534 - Train Acc: 0.7950 - Val Loss: 0.4456 - Val Acc: 0.7874
Epoch 2/10 - Train Loss: 0.3664 - Train Acc: 0.8477 - Val Loss: 0.4306 - Val Acc: 0.8130
Epoch 3/10 - Train Loss: 0.3390 - Train Acc: 0.8614 - Val Loss: 0.3234 - Val Acc: 0.8685
Epoch 4/10 - Train Loss: 0.3203 - Train Acc: 0.8707 - Val Loss: 0.2993 - Val Acc: 0.8764
Epoch 5/10 - Train Loss: 0.3030 - Train Acc: 0.8784 - Val Loss: 0.2818 - Val Acc: 0.8894
Epoch 6/10 - Train Loss: 0.2915 - Train Acc: 0.8843 - Val Loss: 0.3184 - Val Acc: 0.8693
Epoch 7/10 - Train Loss: 0.2793 - Train Acc: 0.8879 - Val Loss: 0.2572 - Val Acc: 0.8963
Epoch 8/10 - Train Loss: 0.2693 - Train Acc: 0.8916 - Val Loss: 0.2601 - Val Acc: 0.8906
Epoch 9/10 - Train Loss: 0.2669 - Train Acc: 0.8943 - Val Loss: 0.2516 - Val Acc: 0.8970
Epoch 10/10 - Train Loss: 0.2588 - Train Acc: 0.8979 - Val Loss: 0.2689 - Val Acc: 0.8915
Test Accuracy:     88.8899
Test Precision:    97.8279
Test Recall:       66.5758
Test F1 Score:     79.2314
